In [ ]:
# =============================================================================
# jobs/16_build_wiski_lookup.py
# Match the telemetry mapping file to the measures in the register, so each measure
# carries its real gauge name, its FFFS id and its WISKI id.
#
# Cadence: manual. Re-run when WISKI_Mapping.csv changes, or after the register gains
# stations (09, 15).
#
# THE KEY. TelemetryID in the file is the equivalent of stationReference, which every
# measure in ea_rt_measure_register carries (it comes from the measure URI). But a station
# has several measures and one TelemetryID can stand for several parameters: Bewdley has a
# flow and a stage, both with TelemetryID 2001. So a row is matched on the station AND what is
# measured. TelemetryParameter (Water Level, Rainfall, Flow ...) and the ParameterID prefix
# (H, Q, P, GW ...) each name a family; if they disagree the row is written as a conflict and
# not matched. A station with one measure is NOT matched on the station alone. The rules and
# the statuses are in utils/telemetry_mapping.py.
#
# WHAT THIS WRITES (unrestricted schema; the file is in the unrestricted volume):
#   ea_rt_telemetry_mapping  every row and EVERY column of the file (the ones it uses by name, and any
#                            other, such as a station name, as src_<column>), with the family each
#                            column names, the measure it matched (measure_uri), a match_status and a note. An
#                            ambiguous row has one line per candidate; use only matched rows as
#                            the link. FFFS_ID is the key to the thresholds.
#   ea_rt_measures_mapped    a view: ea_rt_measure_register with gauge_name (FFFSLocName), fffs_id,
#                            wiski_id, wiski_parameter and every other column of the file where a
#                            measure matched.
#   ea_rt_wiski_lookup       replaces the old table of the same name with a view over the matched
#                            rows that carry a WISKI id. It keeps the old table's columns, in order, so
#                            nothing that reads it breaks, and adds every other column of the file after them. The old
#                            lookup joined on the station alone, so a flow measure could carry a
#                            stage gauge's WISKI id. See the replace_old_table widget.
#
# WIDGETS
#   source_path        the mapping CSV (default config.WISKI_MAPPING_CSV)
#   replace_old_table  "true" drops the old ea_rt_wiski_lookup TABLE so the view can take its name.
#                      Left "false", the table is kept and the view is not created.
#
# Read the printed counts before trusting the table: how many rows matched, and every
# conflict, ambiguity and unmatched row with the reason.
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd

import config
from pyspark.sql.types import StructType, StructField, StringType
from utils.helpers import get_spark, utc_now, table_exists
from utils.telemetry_mapping import prepare_mapping, match_mapping, OUTPUT_FIELDS

spark = get_spark()
now_iso = utc_now().isoformat()

dbutils.widgets.text("source_path", config.WISKI_MAPPING_CSV, "Mapping CSV")
dbutils.widgets.dropdown("replace_old_table", "false", ["false", "true"], "Drop the old ea_rt_wiski_lookup table")
source_path       = dbutils.widgets.get("source_path").strip() or config.WISKI_MAPPING_CSV
replace_old_table = dbutils.widgets.get("replace_old_table").strip().lower() == "true"

if not table_exists(spark, config.TBL_EA_RT_REGISTER):
    raise Exception(f"{config.TBL_EA_RT_REGISTER} does not exist. Run 09 first.")
print(f"Mapping file: {source_path}")


In [ ]:
# =============================================================================
# READ THE FILE AND THE REGISTER
# pandas, not Spark, reads the CSV: it lives in a Volume, and every column is read as text so
# an ID such as 0123 or 4687 is not turned into a number.
# =============================================================================

try:
    raw = pd.read_csv(source_path, dtype=str, encoding="utf-8-sig")
except UnicodeDecodeError:
    raw = pd.read_csv(source_path, dtype=str, encoding="latin-1")
print(f"Read {len(raw):,} rows. Columns: {list(raw.columns)}")

mapping, file_report = prepare_mapping(raw)
print(f"Mapping rows: {file_report['rows']:,}   with no TelemetryID: {file_report['no_telemetry_id']:,}")
if file_report["extra_columns"]:
    print(f"Other columns kept as they are in the file: {file_report['extra_columns']}")
if file_report["wiski_id_in_scientific_notation"]:
    print(f"WARNING: {file_report['wiski_id_in_scientific_notation']:,} WISKI ids are in scientific notation "
          f"(for example 4.53E+08). Excel has already dropped their digits in the file, so they are blanked "
          f"here. Re-save the file with WISKI_ID as text.")

register = spark.table(config.TBL_EA_RT_REGISTER).select(
    "measure_uri", "station_reference", "station_label", "parameter", "qualifier", "period", "unit_name").toPandas()
print(f"Register: {len(register):,} measures at {register['station_reference'].nunique():,} stations")


In [ ]:
# =============================================================================
# MATCH
# =============================================================================

rows, report = match_mapping(mapping, register)

print("\nMatch status of the mapping rows (an ambiguous row counts once per candidate):")
for status, n in sorted(report["statuses"].items(), key=lambda kv: -kv[1]):
    print(f"  {status:<18} {n:>7,}")
print(f"\nMeasures matched: {report['measures_matched']:,} of {len(register):,}")
if report["measures_matched_by_several_rows"]:
    print(f"{report['measures_matched_by_several_rows']:,} measures are matched by more than one row of the file "
          f"(for example two FFFS ids for one gauge). The view takes the lowest FFFS id and WISKI id and counts the FFFS ids.")

for status in ("conflict", "ambiguous", "no_parameter", "unknown_parameter", "no_station"):
    sample = rows[rows["match_status"] == status].drop_duplicates("id").head(10)
    if len(sample):
        print(f"\n{status}: first {len(sample)}")
        print(sample[["telemetry_id", "telemetry_parameter", "parameter_id", "fffs_location_name", "match_note"]].to_string(index=False))

# The case this exists for: a station with several parameters under one TelemetryID.
multi = rows[rows["match_status"] == "matched"].groupby("telemetry_id")["measure_uri"].nunique()
print(f"\nTelemetryIDs matched to more than one measure, each by its own parameter: {int((multi > 1).sum()):,}")


In [ ]:
# =============================================================================
# WRITE: the mapping table, then the two views
# =============================================================================

FILE_EXTRAS = [c for c in mapping.columns if c not in OUTPUT_FIELDS and c not in ("telemetry_family", "code_family")]
FIELDS = OUTPUT_FIELDS + FILE_EXTRAS + ["telemetry_family", "code_family", "match_status", "match_note", "measure_uri",
                                        "station_reference", "station_label", "register_parameter", "register_qualifier",
                                        "period", "unit_name"]
SCHEMA = StructType([StructField(f, StringType(), True) for f in FIELDS + ["mapped_at"]])


def _text(value):
    return None if value is None or (not isinstance(value, str) and pd.isna(value)) else str(value)


records = [tuple(_text(r.get(f)) for f in FIELDS) + (now_iso,) for r in rows.to_dict("records")]
(
    spark.createDataFrame(records, SCHEMA).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.TBL_TELEMETRY_MAPPING)
)
spark.sql(f"""
    COMMENT ON TABLE {config.TBL_TELEMETRY_MAPPING} IS
    'The telemetry mapping file (WISKI_Mapping.csv), every row and every column of it (columns not
     used by name are kept as src_<column>; wiski_id_in_file is the WISKI id as the file holds it),
     each row matched to a measure in
     ea_rt_measure_register on TelemetryID = stationReference AND the parameter it measures.
     match_status: matched (one measure), ambiguous (several candidates, one line each), conflict
     (TelemetryParameter and ParameterID name different parameters), unknown_parameter, no_station,
     no_parameter, no_telemetry_id. Use only matched rows as the link to a measure. fffs_id is the key
     to the thresholds. Rebuilt by 16_build_wiski_lookup.'
""")
print(f"Written {spark.table(config.TBL_TELEMETRY_MAPPING).count():,} rows to {config.TBL_TELEMETRY_MAPPING}")

# Every column of the file that is not already a column of the register comes with the measure.
# Where one measure is matched by several rows, MIN() takes the lowest value of each.
def _q(name):
    return "`" + name + "`"

SRC_COLUMNS = [c for c in FILE_EXTRAS if c.startswith("src_")]     # the file's other columns, as src_<name>
MAPPED_COLS = ["parameter_id", "wiski_xml_parameter", "wiski_xml_qualifier", "wiski_id_in_file"] + SRC_COLUMNS
mapped_select = ",\n               ".join(f"MIN({_q(c)}) AS {_q(c)}" for c in MAPPED_COLS)
spark.sql(f"""
    CREATE OR REPLACE VIEW {config.TBL_MEASURES_MAPPED} AS
    SELECT r.*, m.gauge_name, m.fffs_id, m.n_fffs_ids, m.wiski_id, m.wiski_parameter,
           {", ".join("m." + _q(c) for c in MAPPED_COLS)}
    FROM {config.TBL_EA_RT_REGISTER} r
    LEFT JOIN (
        SELECT measure_uri,
               MIN(fffs_location_name)          AS gauge_name,
               MIN(fffs_id)                     AS fffs_id,
               COUNT(DISTINCT fffs_id)          AS n_fffs_ids,
               MIN(wiski_id)                    AS wiski_id,
               MIN(wiski_parameter)             AS wiski_parameter,
               {mapped_select}
        FROM {config.TBL_TELEMETRY_MAPPING}
        WHERE match_status = 'matched'
        GROUP BY measure_uri
    ) m ON m.measure_uri = r.measure_uri
""")
print(f"View {config.TBL_MEASURES_MAPPED}: "
      f"{spark.table(config.TBL_MEASURES_MAPPED).where('fffs_id IS NOT NULL OR wiski_id IS NOT NULL').count():,} measures carry an FFFS or WISKI id")

# The old table of the same name becomes a view over the matched rows, once the table is dropped.
old_is_table = False
if table_exists(spark, config.TBL_WISKI_LOOKUP):
    old_is_table = spark.catalog.getTable(config.TBL_WISKI_LOOKUP).tableType != "VIEW"
if old_is_table and not replace_old_table:
    print(f"\n{config.TBL_WISKI_LOOKUP} is still the OLD TABLE (it joins on station alone). Not replaced. "
          f"Set the replace_old_table widget to true and re-run to drop it and create the view in its place.")
else:
    if old_is_table:
        spark.sql(f"DROP TABLE {config.TBL_WISKI_LOOKUP}")
        print(f"Dropped the old table {config.TBL_WISKI_LOOKUP}.")
    extra_select = "".join(f",\n               t.{_q(c)}" for c in SRC_COLUMNS)
    spark.sql(f"""
        CREATE OR REPLACE VIEW {config.TBL_WISKI_LOOKUP} AS
        SELECT t.measure_uri,
               t.station_reference,
               t.station_label,
               r.lat,
               r.long,
               t.telemetry_id         AS external_id,
               t.telemetry_parameter  AS external_parameter,
               t.parameter_qualifier,
               t.period,
               t.unit_name            AS unit,
               t.wiski_id             AS internal_id,
               t.wiski_parameter      AS internal_parameter,
               CAST(NULL AS STRING)   AS import_source,
               TRY_CAST(t.wiski_id AS BIGINT) AS wiski_id,
               -- The rest of the file, so the view shows everything the mapping holds.
               t.id                   AS mapping_id,
               t.fffs_id,
               t.fffs_location_name,
               t.parameter_id,
               t.wiski_parameter,
               t.wiski_xml_parameter,
               t.wiski_xml_qualifier,
               t.wiski_id_in_file{extra_select}
        FROM {config.TBL_TELEMETRY_MAPPING} t
        JOIN {config.TBL_EA_RT_REGISTER} r ON r.measure_uri = t.measure_uri
        WHERE t.match_status = 'matched' AND t.wiski_id IS NOT NULL
    """)
    print(f"View {config.TBL_WISKI_LOOKUP}: {spark.table(config.TBL_WISKI_LOOKUP).count():,} rows")

dbutils.notebook.exit("success")
